# Lab 3: End-to-End RAG Pipeline

**Level:** Basic | **Duration:** ~40 minutes

## What You'll Learn
- How to build a complete RAG pipeline from document to answer
- How to split documents into chunks with `RecursiveCharacterTextSplitter`
- How chunk size affects retrieval and answer quality
- How to use LangChain to wire together embedding, retrieval, and generation
- How to get cited answers with source references

## Why This Matters
This is the core RAG pattern you'll use in production. Every advanced technique (HyDE, re-ranking, agentic RAG) builds on top of this foundation.

## Setup

In [ ]:
!pip install -q langchain langchain-openai langchain-chroma chromadb

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your-key-here"

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("Setup complete!")

## Step 1: Create a Sample Document

We'll use a fictional company's employee handbook. This simulates what you'd do with real internal documentation.

In [ ]:
company_handbook = """
# SkyWing Airlines Employee Handbook

## Chapter 1: Company Overview

SkyWing Airlines was founded in 2005 and is headquartered in Amsterdam, Netherlands. We operate a fleet of 85 aircraft serving 120 destinations across Europe, Asia, and North Africa. Our mission is to connect people with affordable, reliable air travel while maintaining the highest safety standards.

As of 2024, SkyWing employs approximately 12,000 people across 30 countries. Our workforce includes pilots, cabin crew, ground staff, maintenance engineers, and corporate employees. We are committed to creating a diverse, inclusive workplace where every employee can thrive.

## Chapter 2: Leave Policy

### Annual Leave
All full-time employees are entitled to 25 working days of paid annual leave per calendar year. Part-time employees receive a pro-rated amount based on their contracted hours. Leave must be requested at least 14 days in advance through the HR portal.

Unused annual leave can be carried over to the next year, up to a maximum of 5 days. Any leave beyond this limit will be forfeited on January 31st. Managers may approve exceptions in cases of extended illness or operational requirements that prevented the employee from taking leave.

### Sick Leave
Employees are entitled to full pay for the first 30 days of sick leave per year. From day 31 to day 90, employees receive 70% of their base salary. After 90 days, long-term disability insurance applies. A medical certificate is required for any absence exceeding 3 consecutive days.

### Parental Leave
Primary caregivers are entitled to 16 weeks of fully paid parental leave. Secondary caregivers receive 6 weeks of fully paid leave. Both can request an additional 12 weeks of unpaid leave. Parental leave must begin within 12 months of the child's birth or adoption date.

## Chapter 3: Remote Work Policy

### Eligibility
Corporate employees may work remotely up to 3 days per week, subject to manager approval. Operational roles (pilots, cabin crew, ground staff, maintenance) are not eligible for remote work due to the nature of their duties.

### Home Office Setup
SkyWing provides a one-time allowance of 500 euros for home office equipment. This covers items such as a desk, chair, monitor, or keyboard. Employees must submit receipts within 90 days of purchase for reimbursement.

### International Remote Work
Employees may request to work from another country for up to 30 consecutive days per year. Requests must be approved by both the employee's manager and the HR department. Tax and social security implications are the employee's responsibility.

## Chapter 4: Performance Reviews

### Review Cycle
Performance reviews are conducted semi-annually, in June and December. The review process includes self-assessment, manager assessment, and a calibration meeting at the department level.

### Rating Scale
Employees are rated on a 5-point scale:
1. Does Not Meet Expectations - Significant improvement needed
2. Partially Meets Expectations - Some areas need improvement
3. Meets Expectations - Solid performance, fulfills role requirements
4. Exceeds Expectations - Consistently delivers above requirements
5. Outstanding - Exceptional performance, role model for others

### Compensation Adjustment
Annual salary adjustments are linked to performance ratings. Employees rated 3 and above are eligible for a merit increase. The average merit increase budget is 4% of total base salary, distributed based on individual ratings. Employees rated 1 or 2 are placed on a Performance Improvement Plan (PIP) and are not eligible for merit increases.

## Chapter 5: Travel Benefits

### Staff Travel
Employees receive unlimited standby travel on SkyWing flights after completing 6 months of service. Standby travel is subject to availability and is confirmed at the gate. Employees must dress in smart casual attire when traveling on staff tickets.

### Buddy Passes
Each employee receives 4 buddy passes per year. These can be used by friends or family who are not eligible for staff travel benefits. Buddy passes provide a 75% discount on the published economy fare. They are non-transferable and expire at the end of the calendar year.

### Interline Agreements
SkyWing has interline agreements with 15 partner airlines, offering employees discounted travel on partner carriers at rates between 50% and 90% off the published fare. Booking must be done through the dedicated staff travel portal.
"""

print(f"Document length: {len(company_handbook)} characters")
print(f"Approximate word count: {len(company_handbook.split())} words")

## Step 2: Split into Chunks

Large documents must be split into smaller chunks for embedding. The `RecursiveCharacterTextSplitter` tries to split at natural boundaries (paragraphs, sentences) before resorting to character-level splits.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_text(company_handbook)

print(f"Split into {len(chunks)} chunks.\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars):")
    print(f"  {chunk[:100]}...")
    print()

## Step 3: Explore Chunk Size Effects

Chunk size is one of the most important parameters in RAG. Let's see how different sizes split the same document.

In [ ]:
for chunk_size in [200, 500, 1000, 2000]:
    splitter_test = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size * 0.1),  # 10% overlap
    )
    test_chunks = splitter_test.split_text(company_handbook)
    avg_len = sum(len(c) for c in test_chunks) / len(test_chunks)
    print(f"chunk_size={chunk_size:>5}: {len(test_chunks):>3} chunks, avg {avg_len:.0f} chars each")

print("\nSmaller chunks = more precise retrieval, but may lose context.")
print("Larger chunks = more context per result, but may include irrelevant text.")
print("500-1000 is a good starting point for most use cases.")

## Step 4: Embed and Store in ChromaDB

Now we embed all chunks and store them in a vector database.

In [ ]:
from langchain_core.documents import Document

# Create Document objects with metadata
docs = [
    Document(
        page_content=chunk,
        metadata={"source": "employee-handbook", "chunk_index": i}
    )
    for i, chunk in enumerate(chunks)
]

# Create vector store
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="handbook_rag",
)

print(f"Stored {len(docs)} document chunks in ChromaDB.")

## Step 5: Test Retrieval

Before connecting to the LLM, let's verify that retrieval returns relevant chunks.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "How many days of annual leave do employees get?"
retrieved_docs = retriever.invoke(query)

print(f"Query: \"{query}\"\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Retrieved chunk {i} (index: {doc.metadata['chunk_index']}) ---")
    print(doc.page_content)
    print()

## Step 6: Build the RAG Chain

Now we connect retrieval to generation. The prompt tells the LLM to answer based only on the retrieved context and cite its sources.

In [ ]:
rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions about the SkyWing Airlines Employee Handbook.

Answer the question based ONLY on the following context. If the context doesn't contain enough information to answer, say "I don't have enough information to answer this question."

After your answer, cite which section(s) of the handbook you used.

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully!")

## Step 7: Ask Questions

Let's test the full pipeline with several questions.

In [ ]:
questions = [
    "How many days of annual leave do employees get?",
    "What is the remote work policy?",
    "How does the performance rating system work?",
    "Can I use staff travel benefits for my friends?",
    "What happens if I'm sick for more than a month?",
]

for q in questions:
    print(f"Q: {q}")
    answer = rag_chain.invoke(q)
    print(f"A: {answer}")
    print("=" * 80)
    print()

## Step 8: Test the Boundaries

What happens when you ask something NOT in the handbook? The system should admit it doesn't know.

In [ ]:
out_of_scope_questions = [
    "What is the company's stock price?",
    "How do I file a complaint about a coworker?",
    "What is the capital of France?",
]

for q in out_of_scope_questions:
    print(f"Q: {q}")
    answer = rag_chain.invoke(q)
    print(f"A: {answer}")
    print("-" * 60)
    print()

## Step 9: Retrieve with Sources

Let's build a version that returns both the answer AND the source chunks, so users can verify.

In [ ]:
def rag_with_sources(question):
    """Run RAG and return both the answer and the source chunks."""
    # Retrieve
    retrieved = retriever.invoke(question)

    # Generate
    context = format_docs(retrieved)
    answer = (rag_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

    return {
        "question": question,
        "answer": answer,
        "sources": [
            {
                "chunk_index": doc.metadata["chunk_index"],
                "content": doc.page_content[:200] + "...",
            }
            for doc in retrieved
        ],
    }

result = rag_with_sources("How long is parental leave?")

print(f"Question: {result['question']}\n")
print(f"Answer: {result['answer']}\n")
print("Sources used:")
for src in result["sources"]:
    print(f"  Chunk {src['chunk_index']}: {src['content']}")
    print()

---

## YOUR TURN: Chunk Size Experiment

Rebuild the pipeline with different chunk sizes and compare answer quality on the same questions.

1. Try `chunk_size=200` (small, precise chunks)
2. Try `chunk_size=1500` (large, context-rich chunks)
3. Ask the same 3 questions with each and compare

Which chunk size gives better answers? Why?

In [ ]:
# YOUR TURN: Experiment with chunk sizes
test_questions = [
    "What is the sick leave policy after 30 days?",
    "Can I work from another country?",
    "How do buddy passes work?",
]

for chunk_size in [200, 1500]:
    print(f"\n{'='*60}")
    print(f"CHUNK SIZE: {chunk_size}")
    print(f"{'='*60}")

    # Rebuild the pipeline with this chunk size
    test_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size * 0.1),
    )
    test_chunks = test_splitter.split_text(company_handbook)
    test_docs = [
        Document(page_content=c, metadata={"chunk_index": i})
        for i, c in enumerate(test_chunks)
    ]

    test_vs = Chroma.from_documents(
        documents=test_docs,
        embedding=embeddings,
        collection_name=f"test_{chunk_size}",
    )
    test_retriever = test_vs.as_retriever(search_kwargs={"k": 3})

    test_chain = (
        {"context": test_retriever | format_docs, "question": RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
    )

    print(f"Chunks: {len(test_chunks)}")
    for q in test_questions:
        answer = test_chain.invoke(q)
        print(f"\nQ: {q}")
        print(f"A: {answer[:200]}..." if len(answer) > 200 else f"A: {answer}")

## Key Takeaways

1. **The RAG pipeline has three stages:** split/embed/store (indexing), retrieve (search), generate (LLM).
2. **Chunk size is critical** — too small loses context, too large dilutes relevance. Start with 500-1000 chars.
3. **Chunk overlap** prevents information loss at boundaries.
4. **The prompt matters** — instruct the LLM to use only the context and cite sources.
5. **Retrieval quality > Generation quality** — if you retrieve the wrong chunks, the LLM can't save you.

**Next:** In Lab 4, we'll improve retrieval with HyDE (Hypothetical Document Embeddings).